In [1]:
!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta(cache_wheel_on_google_drive=False, serialization=False, mirror=0)
import pyrosetta; pyrosetta.init()
from pyrosetta import *
from pyrosetta.teaching import *


Note that USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE.
See https://github.com/RosettaCommons/rosetta/blob/main/LICENSE.md or email license@uw.edu for details.

Looking for compatible PyRosetta wheel file at google-drive/PyRosetta/colab.bin//wheels...

Resolving west.rosettacommons.org (west.rosettacommons.org)... 128.95.160.135, 128.95.160.134, 2607:4000:406::160:134, ...
Connecting to west.rosettacommons.org (west.rosettacommons.org)|128.95.160.135|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
--2026-09-14 07:55:57--  https://west.rosettacommons.org/pyrosetta/quarterly/release/pyrosetta-0-cp313-cp313-linux_x86_64.whl

HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
Saving to: ‘/PyRosetta/colab.bin/tmp19wpo1ek/pyrosetta-0-cp313-cp313-linux_x86_64.whl’

pyrosetta-0-cp313-c 100%[===================>]   1.55G  24.7MB/s    in 77s     

202

In [2]:
import os
import shutil
import numpy as np
import pandas as pd

import pyrosetta
from pyrosetta import rosetta

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

In [3]:
pyrosetta.init("-mute all")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

In [4]:
scorefxn = pyrosetta.create_score_function("ref2015")

In [37]:
pdb_files = []
pdb_name_map = {}

upload_button = widgets.Button(
    description="Upload PDB files",
    button_style="info",
    icon="upload"
)

upload_output = widgets.Output()


def upload_pdbs(_):
    global pdb_files, pdb_name_map

    with upload_output:
        clear_output()

        uploaded = files.upload()

        pdb_files.clear()
        pdb_name_map.clear()

        n = 1

        for original_name in uploaded.keys():

            if not original_name.lower().endswith(".pdb"):
                continue

            safe_name = f"interface_input_{n:03d}.pdb"
            safe_path = os.path.join("/content", safe_name)

            shutil.copy2(original_name, safe_path)

            pdb_files.append(safe_path)

            # Remember original filename
            pdb_name_map[safe_path] = original_name

            n += 1

        print(f"{len(pdb_files)} PDB file(s) ready:\n")

        for path in pdb_files:
            print(
                f"{pdb_name_map[path]}  ->  "
                f"{os.path.basename(path)}"
            )


upload_button.on_click(upload_pdbs)

display(upload_button, upload_output)

Button(button_style='info', description='Upload PDB files', icon='upload', style=ButtonStyle())

Output()

In [38]:
def run_fastrelax(pose, repeats=1):

    # Use ref2015 for standard all-atom FastRelax
    relax_scorefxn = pyrosetta.create_score_function(
        "ref2015"
    )

    # Create FastRelax
    relax = rosetta.protocols.relax.FastRelax()

    relax.set_scorefxn(relax_scorefxn)

    # Allow backbone and side-chain movement
    movemap = pyrosetta.MoveMap()

    movemap.set_bb(True)
    movemap.set_chi(True)

    # Keep chains from moving relative to one another
    movemap.set_jump(False)

    relax.set_movemap(movemap)

    # FastRelax internally performs multiple rounds.
    # repeats here means independent applications.
    for _ in range(repeats):
        relax.apply(pose)

    return pose

In [39]:
interface_widget = widgets.Text(
    value="A_B",
    description="Interface:",
    placeholder="A_B or AB_C",
    layout=widgets.Layout(width="350px")
)

pack_input_widget = widgets.Checkbox(
    value=False,
    description="Repack bound interface"
)

pack_separated_widget = widgets.Checkbox(
    value=True,
    description="Repack separated partners"
)

packstat_widget = widgets.Checkbox(
    value=False,
    description="Calculate packstat"
)

interface_sc_widget = widgets.Checkbox(
    value=False,
    description="Calculate shape complementarity"
)

pack_rounds_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description="Pack rounds:",
    continuous_update=False
)

display(
    interface_widget,
    pack_input_widget,
    pack_separated_widget,
    packstat_widget,
    interface_sc_widget,
    pack_rounds_widget
)

Text(value='A_B', description='Interface:', layout=Layout(width='350px'), placeholder='A_B or AB_C')

Checkbox(value=False, description='Repack bound interface')

Checkbox(value=True, description='Repack separated partners')

Checkbox(value=False, description='Calculate packstat')

Checkbox(value=False, description='Calculate shape complementarity')

IntSlider(value=1, continuous_update=False, description='Pack rounds:', max=5, min=1)

In [40]:
relax_widget = widgets.Checkbox(
    value=False,
    description="Run FastRelax before InterfaceAnalyzer"
)

relax_repeats_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description="FastRelax repeats:",
    continuous_update=False
)

display(
    relax_widget,
    relax_repeats_widget
)

Checkbox(value=False, description='Run FastRelax before InterfaceAnalyzer')

IntSlider(value=1, continuous_update=False, description='FastRelax repeats:', max=5, min=1)

In [41]:
def extract_all_interface_metrics(iam):

    results = {}

    # Find every getter exposed by this particular
    # PyRosetta InterfaceAnalyzerMover build.
    getter_names = sorted(
        name
        for name in dir(iam)
        if name.startswith("get_")
    )

    for name in getter_names:

        method = getattr(iam, name)

        if not callable(method):
            continue

        try:
            value = method()

            # Numeric values
            if isinstance(
                value,
                (
                    int,
                    float,
                    bool,
                    np.integer,
                    np.floating
                )
            ):
                results[name] = value

            # Strings
            elif isinstance(value, str):
                results[name] = value

            # Other Rosetta objects
            else:
                results[name] = str(value)

        except Exception as e:

            # Some getters may only work when a particular
            # optional calculation has been enabled.
            results[name] = (
                f"ERROR: {type(e).__name__}: {e}"
            )

    return results

In [42]:
def clean_metric_name(name):

    mapping = {

        "get_interface_dG":
            "Interface_dG_REU",

        "get_interface_delta_sasa":
            "Interface_dSASA_A2",

        "get_interface_delta_hbond_unsat":
            "Interface_delta_hbond_unsat",

        "get_interface_packstat":
            "Interface_packstat",

        "get_num_interface_residues":
            "N_interface_residues",

        "get_separated_interface_energy":
            "Separated_interface_energy_REU",

        "get_separated_interface_energy_ratio":
            "Separated_interface_energy_ratio",

        "get_crossterm_interface_energy":
            "Crossterm_interface_energy_REU",

        "get_crossterm_interface_energy_ratio":
            "Crossterm_interface_energy_ratio",

        "get_gly_interface_energy":
            "Gly_interface_energy_REU",

        "get_interface_set":
            "Interface_residue_set",

        "get_pymol_sel_interface":
            "PyMOL_interface_selection",
    }

    return mapping.get(name, name)

In [43]:
def extract_interface_metrics(iam):

    raw_metrics = extract_all_interface_metrics(iam)

    results = {}

    for name, value in raw_metrics.items():

        clean_name = clean_metric_name(name)

        results[clean_name] = value

    return results

In [44]:
def run_interface_analyzer(pdb_path, interface):

    # Load PDB
    pose = pyrosetta.pose_from_pdb(pdb_path)

    # Convert "Z_I" -> DockingPartners
    partners = (
        rosetta.core.pose.DockingPartners
        .docking_partners_from_string(interface)
    )

    # Create InterfaceAnalyzerMover
    iam = rosetta.protocols.analysis.InterfaceAnalyzerMover()

    # Set interface
    iam.set_interface(partners)

    # Score function
    iam.set_scorefunction(scorefxn)

    # Packing options
    iam.set_pack_input(
        bool(pack_input_widget.value)
    )

    iam.set_pack_separated(
        bool(pack_separated_widget.value)
    )

    iam.set_pack_rounds(
        int(pack_rounds_widget.value)
    )

    # Optional calculations
    iam.set_compute_packstat(
        bool(packstat_widget.value)
    )

    iam.set_compute_interface_sc(
        bool(interface_sc_widget.value)
    )

    # Run InterfaceAnalyzer
    iam.apply(pose)

    return pose, iam

In [45]:
run_button = widgets.Button(
    description="Run InterfaceAnalyzer",
    button_style="success",
    icon="play"
)

run_output = widgets.Output()

latest_interface_results = pd.DataFrame()


def run_all_interfaces(_):

    global latest_interface_results

    with run_output:

        clear_output()

        # =========================================================
        # CHECK INPUTS
        # =========================================================

        if not pdb_files:
            print("Please upload at least one PDB file.")
            return

        interface = interface_widget.value.strip()

        if not interface:
            print(
                "Please specify an interface, "
                "for example Z_I."
            )
            return

        print("=" * 70)
        print("INTERFACE ANALYSIS")
        print("=" * 70)

        print(
            f"Number of PDB files: {len(pdb_files)}"
        )

        print(
            f"Interface: {interface}"
        )

        if relax_widget.value:
            print(
                "FastRelax: ENABLED "
                f"({relax_repeats_widget.value} repeat(s))"
            )
        else:
            print("FastRelax: disabled")

        print(
            f"Repack bound interface: "
            f"{pack_input_widget.value}"
        )

        print(
            f"Repack separated interface: "
            f"{pack_separated_widget.value}"
        )

        print()

        # =========================================================
        # RESULTS STORAGE
        # =========================================================

        results = []

        # =========================================================
        # PROCESS EVERY PDB
        # =========================================================

        for pdb_index, pdb_path in enumerate(
            pdb_files,
            start=1
        ):

            original_name = pdb_name_map.get(
                pdb_path,
                os.path.basename(pdb_path)
            )

            print()
            print("=" * 70)
            print(
                f"PDB {pdb_index}/{len(pdb_files)}"
            )
            print(
                f"Input: {original_name}"
            )
            print("=" * 70)

            # -----------------------------------------------------
            # Make sure file still exists
            # -----------------------------------------------------

            if not os.path.exists(pdb_path):

                print(
                    f"FAILED: File not found: {pdb_path}"
                )

                results.append({
                    "PDB": original_name,
                    "Interface": interface,
                    "Status": "File not found"
                })

                continue

            try:

                # =================================================
                # LOAD PDB
                # =================================================

                print("Loading PDB...")

                pose = pyrosetta.pose_from_pdb(
                    pdb_path
                )

                print(
                    f"Loaded {pose.total_residue()} "
                    f"residues."
                )

                # =================================================
                # OPTIONAL FASTRELAX
                # =================================================

                relaxed_path = None

                if relax_widget.value:

                    print()
                    print("Running FastRelax...")

                    pose = run_fastrelax(
                        pose,
                        repeats=(
                            relax_repeats_widget.value
                        )
                    )

                    print(
                        "FastRelax complete."
                    )

                    # ---------------------------------------------
                    # Save relaxed structure
                    # ---------------------------------------------

                    base_name = os.path.splitext(
                        original_name
                    )[0]

                    relaxed_filename = (
                        f"{base_name}.relaxed.pdb"
                    )

                    relaxed_path = os.path.join(
                        "/content",
                        relaxed_filename
                    )

                    pose.dump_pdb(
                        relaxed_path
                    )

                    print(
                        f"Relaxed PDB saved as: "
                        f"{relaxed_filename}"
                    )

                # =================================================
                # CREATE DOCKING PARTNERS
                # =================================================

                partners = (
                    rosetta.core.pose.DockingPartners
                    .docking_partners_from_string(
                        interface
                    )
                )

                # =================================================
                # CREATE INTERFACE ANALYZER
                # =================================================

                iam = (
                    rosetta.protocols.analysis
                    .InterfaceAnalyzerMover()
                )

                iam.set_interface(
                    partners
                )

                iam.set_scorefunction(
                    scorefxn
                )

                # -------------------------------------------------
                # Packing settings
                # -------------------------------------------------

                iam.set_pack_input(
                    bool(
                        pack_input_widget.value
                    )
                )

                iam.set_pack_separated(
                    bool(
                        pack_separated_widget.value
                    )
                )

                iam.set_pack_rounds(
                    int(
                        pack_rounds_widget.value
                    )
                )

                # -------------------------------------------------
                # Optional calculations
                # -------------------------------------------------

                iam.set_compute_packstat(
                    bool(
                        packstat_widget.value
                    )
                )

                iam.set_compute_interface_sc(
                    bool(
                        interface_sc_widget.value
                    )
                )

                # =================================================
                # RUN INTERFACE ANALYZER
                # =================================================

                print()
                print(
                    "Running InterfaceAnalyzer..."
                )

                iam.apply(pose)

                print(
                    "InterfaceAnalyzer complete."
                )

                # =================================================
                # EXTRACT ALL METRICS
                # =================================================

                metrics = extract_interface_metrics(
                    iam
                )

                # =================================================
                # CREATE RESULT ROW
                # =================================================

                row = {
                    "PDB": original_name,
                    "Interface": interface,
                    "N_residues":
                        pose.total_residue(),
                    "FastRelax":
                        bool(
                            relax_widget.value
                        ),
                    "Relaxed_PDB":
                        (
                            os.path.basename(
                                relaxed_path
                            )
                            if relaxed_path
                            else ""
                        ),
                    "Status": "Success"
                }

                # Add all InterfaceAnalyzer metrics
                row.update(metrics)

                results.append(row)

                # =================================================
                # PRINT IMPORTANT METRICS
                # =================================================

                print()
                print(
                    "-" * 50
                )
                print("Key results:")

                if "Interface_dG_REU" in metrics:
                    print(
                        "  Interface dG:       "
                        f"{metrics['Interface_dG_REU']:.3f}"
                    )

                if "Interface_dSASA_A2" in metrics:
                    print(
                        "  Interface dSASA:    "
                        f"{metrics['Interface_dSASA_A2']:.2f} A^2"
                    )

                if "N_interface_residues" in metrics:
                    print(
                        "  Interface residues: "
                        f"{metrics['N_interface_residues']}"
                    )

                if (
                    "Separated_interface_energy_REU"
                    in metrics
                ):
                    print(
                        "  Separated energy:   "
                        f"{metrics['Separated_interface_energy_REU']:.3f}"
                    )

                if (
                    "Crossterm_interface_energy_REU"
                    in metrics
                ):
                    print(
                        "  Crossterm energy:   "
                        f"{metrics['Crossterm_interface_energy_REU']:.3f}"
                    )

                if "Interface_packstat" in metrics:
                    print(
                        "  Packstat:            "
                        f"{metrics['Interface_packstat']}"
                    )

                print(
                    "-" * 50
                )

            # =====================================================
            # HANDLE FAILURE FOR THIS PDB
            # =====================================================

            except Exception as e:

                error_message = (
                    f"{type(e).__name__}: {e}"
                )

                print()
                print(
                    f"FAILED: {error_message}"
                )

                results.append({
                    "PDB": original_name,
                    "Interface": interface,
                    "N_residues": np.nan,
                    "FastRelax":
                        bool(
                            relax_widget.value
                        ),
                    "Relaxed_PDB": "",
                    "Status":
                        f"Failed: {error_message}"
                })

        # =========================================================
        # CREATE FINAL RESULTS TABLE
        # =========================================================

        latest_interface_results = pd.DataFrame(
            results
        )

        print()
        print("=" * 70)
        print("INTERFACE ANALYSIS COMPLETE")
        print("=" * 70)

        print(
            f"Completed {len(results)} "
            f"of {len(pdb_files)} PDB file(s)."
        )

        # Display table
        display(
            latest_interface_results
        )

        # =========================================================
        # SAVE RESULTS
        # =========================================================

        output_csv = (
            "interface_analyzer_results.csv"
        )

        latest_interface_results.to_csv(
            output_csv,
            index=False
        )

        print()
        print(
            f"Results saved to: {output_csv}"
        )


run_button.on_click(
    run_all_interfaces
)

display(
    run_button,
    run_output
)

Button(button_style='success', description='Run InterfaceAnalyzer', icon='play', style=ButtonStyle())

Output()